# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import os

data_path = './data'


In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

print("Training data sample:")
print(df_train[['src', 'tgt']].head())
print(f"\nDataset sizes: Train={len(df_train)}, Eval={len(df_eval)}")


Training data sample:
           src   tgt
0  14*(43+20)=   882
1     (6+1)*5=    35
2    13+32+29=    74
3   31*(3-11)=  -248
4     24*49+1=  1177

Dataset sizes: Train=2369250, Eval=263250


# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [4]:
char_to_id = {}
id_to_char = {}

# Add special tokens
char_to_id['<pad>'] = 0
char_to_id['<eos>'] = 1
char_to_id['<unk>'] = 2

id_to_char[0] = '<pad>'
id_to_char[1] = '<eos>'
id_to_char[2] = '<unk>'

# Build vocabulary from all characters in training data
idx = 3
all_chars = ''.join(df_train['src'])
for char in sorted(set(all_chars)):
    if char not in char_to_id:
        char_to_id[char] = idx
        id_to_char[idx] = char
        idx += 1

vocab_size = len(char_to_id)
print(f'Vocab size: {vocab_size}')
print('Character mappings (first 20):')
for i in range(min(20, vocab_size)):
    print(f'  {id_to_char[i]}: {i}')


Vocab size: 19
Character mappings (first 20):
  <pad>: 0
  <eos>: 1
  <unk>: 2
  (: 3
  ): 4
  *: 5
  +: 6
  -: 7
  0: 8
  1: 9
  2: 10
  3: 11
  4: 12
  5: 13
  6: 14
  7: 15
  8: 16
  9: 17
  =: 18


In [ ]:
batch_size = 64
epochs = 30
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1


In [6]:
def char_to_id_with_unk(char):
    """Convert character to ID, use <unk> for unknown characters"""
    return char_to_id.get(char, char_to_id['<unk>'])

def create_training_sequences(row):
    """
    Create input/output sequences for seq2seq arithmetic learning.
    Input: equation with '=' (e.g., "1+2=")
    Output: answer only (e.g., "3<eos>")
    """
    src = row['src']
    tgt = row['tgt']
    
    if not src.endswith('='):
        src = src + '='
    
    input_ids = [char_to_id_with_unk(c) for c in src]
    label_ids = [char_to_id_with_unk(c) for c in tgt] + [char_to_id['<eos>']]
    equals_pos = src.index('=')
    
    return input_ids, label_ids, equals_pos

# Create training sequences
df_train['char_id_list'], df_train['label_id_list'], df_train['equals_pos'] = \
    zip(*df_train.apply(create_training_sequences, axis=1))

# Show examples
print("=== Data Preprocessing Examples ===\n")
for idx in range(min(5, len(df_train))):
    row = df_train.iloc[idx]
    input_chars = [id_to_char[id] for id in row['char_id_list']]
    label_chars = [id_to_char[id] for id in row['label_id_list']]
    
    print(f"Example {idx + 1}:")
    print(f"  Equation: {row['src']}")
    print(f"  Answer: {row['tgt']}")
    print(f"  Input seq: {''.join(input_chars)}")
    print(f"  Label seq: {''.join(label_chars)}")
    print()

print("=== Dataset Statistics ===")
print(f"Total samples: {len(df_train)}")
print(f"Avg equation length: {df_train['char_id_list'].apply(len).mean():.2f}")
print(f"Avg answer length: {df_train['label_id_list'].apply(len).mean():.2f}")


=== Data Preprocessing Examples ===

Example 1:
  Equation: 14*(43+20)=
  Answer: 882
  Input seq: 14*(43+20)=
  Label seq: 882<eos>

Example 2:
  Equation: (6+1)*5=
  Answer: 35
  Input seq: (6+1)*5=
  Label seq: 35<eos>

Example 3:
  Equation: 13+32+29=
  Answer: 74
  Input seq: 13+32+29=
  Label seq: 74<eos>

Example 4:
  Equation: 31*(3-11)=
  Answer: -248
  Input seq: 31*(3-11)=
  Label seq: -248<eos>

Example 5:
  Equation: 24*49+1=
  Answer: 1177
  Input seq: 24*49+1=
  Label seq: 1177<eos>

=== Dataset Statistics ===
Total samples: 2369250
Avg equation length: 9.53
Avg answer length: 3.90


# Hyperparameters

Define all hyperparameters before building the model.

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|


# Data Batching

- Use `torch.utils.data.Dataset` to create a data generation tool called `dataset`.
- Then, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.


In [7]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, index):
        row = self.sequences.iloc[index]
        x = row['char_id_list']
        y = row['label_id_list']
        equals_pos = row['equals_pos'] if 'equals_pos' in row else -1
        return x, y, equals_pos

def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_equals_pos = [data[2] for data in batch]
    
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    encoder_mask = torch.arange(pad_batch_x.size(1)).unsqueeze(0) < batch_x_lens.unsqueeze(1)
    encoder_mask = encoder_mask.float()
    
    equals_mask = torch.zeros_like(pad_batch_x, dtype=torch.float32)
    for i, pos in enumerate(batch_equals_pos):
        if pos >= 0 and pos < pad_batch_x.size(1):
            equals_mask[i, pos] = 1.0
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens, encoder_mask, equals_mask


In [8]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list', 'equals_pos']])

dl_train = torch.utils.data.DataLoader(ds_train, 
                                       batch_size=batch_size,
                                       shuffle=True,
                                       collate_fn=collate_fn,
                                       num_workers=0)

print(f"DataLoader created: {len(dl_train)} batches")

# Verify with one batch
batch_x, batch_y, batch_x_lens, batch_y_lens, encoder_mask, equals_mask = next(iter(dl_train))
print(f"Batch shapes: x={batch_x.shape}, y={batch_y.shape}")

sample_x = batch_x[0][:batch_x_lens[0]]
sample_y = batch_y[0][:batch_y_lens[0]]
input_str = ''.join([id_to_char[int(id)] for id in sample_x])
label_str = ''.join([id_to_char[int(id)] for id in sample_y])

print(f"Sample: Input='{input_str}', Label='{label_str}'")


DataLoader created: 37020 batches
Batch shapes: x=torch.Size([64, 11]), y=torch.Size([64, 6])
Sample: Input='(29+23)-35=', Label='17<eos>'


# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [9]:
torch.manual_seed(2)

device = torch.device('cpu')

if torch.cuda.is_available():
    try:
        device = torch.device('cuda')
        test_tensor = torch.randn(2, 3).to(device)
        print(f"Using CUDA GPU (device: {device})")
    except Exception as e:
        print(f"CUDA initialization failed: {e}")
        device = torch.device('cpu')
        print(f"Falling back to CPU")
else:
    print(f"Using CPU (device: {device})")


Using CUDA GPU (device: cuda)


In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        
        self.encoder_embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                                     embedding_dim=embed_dim,
                                                     padding_idx=char_to_id['<pad>'])
        
        self.encoder_rnn = torch.nn.LSTM(input_size=embed_dim,
                                         hidden_size=hidden_dim,
                                         num_layers=2,
                                         batch_first=True)
        
        self.decoder_embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                                     embedding_dim=embed_dim,
                                                     padding_idx=char_to_id['<pad>'])
        
        self.decoder_rnn = torch.nn.LSTM(input_size=embed_dim,
                                         hidden_size=hidden_dim,
                                         num_layers=2,
                                         batch_first=True)
        
        self.output_layer = torch.nn.Sequential(
            torch.nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features=hidden_dim, out_features=vocab_size)
        )
    
    def forward(self, encoder_input, encoder_lens, decoder_input, decoder_lens):
        """Seq2Seq forward pass with teacher forcing."""
        encoder_output, (hidden, cell) = self.encode(encoder_input, encoder_lens)
        decoder_output = self.decode(decoder_input, decoder_lens, hidden, cell)
        return decoder_output
    
    def encode(self, batch_x, batch_x_lens):
        """Encode the equation to get context."""
        batch_x = self.encoder_embedding(batch_x)
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens.cpu(),
                                                          batch_first=True,
                                                          enforce_sorted=False)
        output, (hidden, cell) = self.encoder_rnn(batch_x)
        return output, (hidden, cell)
    
    def decode(self, batch_y, batch_y_lens, hidden, cell):
        """Decode to generate answer using teacher forcing."""
        batch_y = self.decoder_embedding(batch_y)
        batch_y = torch.nn.utils.rnn.pack_padded_sequence(batch_y,
                                                          batch_y_lens.cpu(),
                                                          batch_first=True,
                                                          enforce_sorted=False)
        output, _ = self.decoder_rnn(batch_y, (hidden, cell))
        output, _ = torch.nn.utils.rnn.pad_packed_sequence(output, batch_first=True)
        output = self.output_layer(output)
        return output
    
    def generator(self, equation, max_len=20):
        """Generate answer for a given equation."""
        self.eval()
        
        if not equation:
            return ['<eos>']
        
        encoder_input = [char_to_id.get(c, char_to_id['<unk>']) for c in equation]
        encoder_tensor = torch.LongTensor([encoder_input]).to(device)
        encoder_lens = torch.LongTensor([len(encoder_input)])
        
        with torch.no_grad():
            _, (hidden, cell) = self.encode(encoder_tensor, encoder_lens)
            generated_ids = []
            decoder_input = torch.LongTensor([[char_to_id['<eos>']]]).to(device)
            
            for _ in range(max_len):
                embedded = self.decoder_embedding(decoder_input)
                output, (hidden, cell) = self.decoder_rnn(embedded, (hidden, cell))
                logits = self.output_layer(output[:, -1, :])
                next_token = torch.argmax(logits, dim=-1).item()
                
                if next_token == char_to_id['<eos>']:
                    generated_ids.append(next_token)
                    break
                
                generated_ids.append(next_token)
                decoder_input = torch.LongTensor([[next_token]]).to(device)
            
            return [id_to_char.get(ch_id, '<unk>') for ch_id in generated_ids]


In [11]:
model = CharRNN(vocab_size, embed_dim, hidden_dim)
print(f"Model created: vocab={vocab_size}, embed={embed_dim}, hidden={hidden_dim}")


Model created: vocab=19, embed=256, hidden=256


# Training

The training loop:
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss.
        3. Use `loss.backward()` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_()` to limit the gradient values.
        5. Use `optimizer.step()` to update the model (backpropagation).
    2. After each epoch, perform validation on a sample of the eval dataset.
2. Save the best model based on validation accuracy.


In [12]:
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=char_to_id['<pad>'],
    reduction='mean'
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print(f"Optimizer: Adam (lr={lr}), Loss: CrossEntropyLoss, Grad clip={grad_clip}")


Optimizer: Adam (lr=0.001), Loss: CrossEntropyLoss, Grad clip=1


In [13]:
from tqdm import tqdm

VALIDATION_SAMPLE_SIZE = 5000

model = model.to(device)
model.train()

print("=" * 60)
print("Starting Seq2Seq Training for Arithmetic")
print("=" * 60)

i = 0
train_losses = []
best_em = 0.0
best_model_state = None

for epoch in range(1, epochs+1):
    epoch_losses = []
    
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens, encoder_mask, equals_mask in bar:
        optimizer.zero_grad()
        
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        encoder_mask = encoder_mask.to(device)
        equals_mask = equals_mask.to(device)
        
        decoder_input = batch_y[:, :-1]
        decoder_lens = torch.clamp(batch_y_lens - 1, min=1)
        decoder_target = batch_y[:, 1:]
        
        batch_pred_y = model(batch_x, batch_x_lens, decoder_input, decoder_lens)
        
        pred_len = batch_pred_y.size(1)
        target_len = decoder_target.size(1)
        
        if pred_len != target_len:
            if pred_len > target_len:
                batch_pred_y = batch_pred_y[:, :target_len, :]
            else:
                decoder_target = decoder_target[:, :pred_len]
        
        batch_pred_y_reshaped = batch_pred_y.reshape(-1, vocab_size)
        batch_y_reshaped = decoder_target.reshape(-1)
        
        loss = criterion(batch_pred_y_reshaped, batch_y_reshaped)
        
        loss.backward()
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)
        optimizer.step()
        
        epoch_losses.append(loss.item())
        train_losses.append(loss.item())
        
        i += 1
        if i % 50 == 0:
            avg_loss = np.mean(epoch_losses[-50:])
            bar.set_postfix(loss=f"{avg_loss:.6f}")
    
    epoch_loss = np.mean(epoch_losses)
    print(f"\nEpoch {epoch} - Loss: {epoch_loss:.6f} (min={min(epoch_losses):.6f}, max={max(epoch_losses):.6f})")
    
    # Validation
    print(f"Running validation...")
    model.eval()
    matched = 0
    total = 0
    error_samples = []
    
    eval_sample_size = min(VALIDATION_SAMPLE_SIZE, len(df_eval))
    df_eval_sample = df_eval.sample(n=eval_sample_size, random_state=42)
    
    with torch.no_grad():
        for _, row in tqdm(df_eval_sample.iterrows(), desc=f"Validation", total=len(df_eval_sample)):
            equation = row['src']
            expected_answer = str(row['tgt'])
            
            try:
                prediction_chars = model.generator(equation)
                predicted_answer = ''.join(prediction_chars).replace('<eos>', '').strip()
                
                if predicted_answer == expected_answer:
                    matched += 1
                else:
                    if len(error_samples) < 5:
                        error_samples.append((equation, expected_answer, predicted_answer))
                        
            except Exception as e:
                if len(error_samples) < 5:
                    error_samples.append((equation, expected_answer, f"Error: {str(e)}"))
            
            total += 1
    
    em = matched / total if total > 0 else 0
    print(f"Validation EM: {em:.4f} ({matched}/{total})")
    
    if error_samples:
        print(f"Sample errors:")
        for err_idx, (eq, expected, predicted) in enumerate(error_samples[:5], 1):
            print(f"  {err_idx}. {eq} | Expected: {expected} | Got: {predicted}")
    
    if em > best_em:
        best_em = em
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"✓ New best model! EM: {best_em:.4f}")
    
    print("=" * 60 + "\n")
    model.train()

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(device)
    print(f"Loaded best model with EM: {best_em:.4f}")

print("=" * 60)
print("Training Complete!")
print("=" * 60)
print(f"Final loss: {train_losses[-1]:.6f}, Average: {np.mean(train_losses):.6f}")
print(f"Best validation EM: {best_em:.4f}")


Starting Seq2Seq Training for Arithmetic


Train epoch 1: 100%|██████████| 37020/37020 [12:31<00:00, 49.24it/s, loss=0.323413]



Epoch 1 - Loss: 0.528868 (min=0.168120, max=2.949989)
Running validation...


Validation: 100%|██████████| 5000/5000 [00:06<00:00, 806.76it/s]


Validation EM: 0.0040 (20/5000)
Sample errors:
  1. 19+49+2= | Expected: 70 | Got: 92
  2. (37*46)-22= | Expected: 1680 | Got: 080
  3. 35*(26-43)= | Expected: -595 | Got: 85
  4. 39-(9+4)= | Expected: 26 | Got: 0
  5. 2-(25+44)= | Expected: -67 | Got: 7
✓ New best model! EM: 0.0040



...

Validation EM: 0.0042 (21/5000)
Sample errors:
  1. 19+49+2= | Expected: 70 | Got: 8
  2. (37*46)-22= | Expected: 1680 | Got: 020
  3. 35*(26-43)= | Expected: -595 | Got: 55
  4. 39-(9+4)= | Expected: 26 | Got: 0
  5. 2-(25+44)= | Expected: -67 | Got: 7



Train epoch 10: 100%|██████████| 37020/37020 [12:33<00:00, 49.14it/s, loss=0.069632]



Epoch 10 - Loss: 0.073119 (min=0.005788, max=0.432839)
Running validation...


Validation: 100%|██████████| 5000/5000 [00:10<00:00, 481.31it/s]


Validation EM: 0.0034 (17/5000)
Sample errors:
  1. 19+49+2= | Expected: 70 | Got: 8
  2. (37*46)-22= | Expected: 1680 | Got: 060
  3. 35*(26-43)= | Expected: -595 | Got: 95
  4. 39-(9+4)= | Expected: 26 | Got: 0
  5. 2-(25+44)= | Expected: -67 | Got: 6

Loaded best model with EM: 0.0060
Training Complete!
Final loss: 0.030507, Average: 0.157604
Best validation EM: 0.0060


In [14]:
print("Running full evaluation on entire eval set...")
print(f"Total samples: {len(df_eval)}")

model.eval()
matched = 0
total = 0
error_samples = []

with torch.no_grad():
    for _, row in tqdm(df_eval.iterrows(), desc="Full Evaluation", total=len(df_eval)):
        equation = row['src']
        expected_answer = str(row['tgt'])
        
        try:
            prediction_chars = model.generator(equation)
            predicted_answer = ''.join(prediction_chars).replace('<eos>', '').strip()
            
            if predicted_answer == expected_answer:
                matched += 1
            else:
                if len(error_samples) < 20:
                    error_samples.append((equation, expected_answer, predicted_answer))
        except Exception as e:
            if len(error_samples) < 20:
                error_samples.append((equation, expected_answer, f"Error: {str(e)}"))
        
        total += 1

full_em = matched / total if total > 0 else 0
print(f"\n{'='*60}")
print(f"Full Evaluation Results")
print(f"{'='*60}")
print(f"Exact Match: {full_em:.4f} ({matched}/{total})")
print(f"Accuracy: {full_em*100:.2f}%")

if error_samples:
    print(f"\n{'='*60}")
    print(f"Sample Errors (showing first 10):")
    print(f"{'='*60}")
    for idx, (eq, expected, predicted) in enumerate(error_samples[:10], 1):
        print(f"{idx:2d}. {eq:20s} | Expected: {expected:10s} | Got: {predicted}")


Running full evaluation on entire eval set...
Total samples: 263250


Full Evaluation: 100%|██████████| 263250/263250 [04:37<00:00, 949.97it/s] 


Full Evaluation Results
Exact Match: 0.0049 (1288/263250)
Accuracy: 0.49%

Sample Errors (showing first 10):
 1. 48+43+34=            | Expected: 125        | Got: 01
 2. 30-(48+13)=          | Expected: -31        | Got: 1
 3. (21*31)+10=          | Expected: 661        | Got: 91
 4. 2-27-10=             | Expected: -35        | Got: 1
 5. (15*20)+24=          | Expected: 324        | Got: 04
 6. 10*35*26=            | Expected: 9100       | Got: 920
 7. 2-(45+30)=           | Expected: -73        | Got: 13
 8. 43*41*3=             | Expected: 5289       | Got: 999
 9. 5-13*17=             | Expected: -216       | Got: 06
10. (1+6)-5=             | Expected: 2          | Got: 
